# Generative Agents: Full Simulation (Claude API)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tiraphap/Gen-Agent-Lectures-at-Econ-TU/blob/master/teaching/notebooks/full_simulation_claude.ipynb)

Notebook นี้รวมเนื้อหาจาก **Notebook 01-03** เข้าด้วยกัน พร้อมใช้ **Claude API** เป็น LLM:

1. **Memory Stream** — ระบบความทรงจำ + สูตร Retrieval
2. **Cognitive Loop** — วงจรการคิด 5 ขั้นตอน
3. **Multi-Agent Retail Simulation** — จำลองลูกค้า 3 คน

---
**ผศ.ดร.ถิรภาพ ฟักทอง** | คณะเศรษฐศาสตร์ มหาวิทยาลัยธรรมศาสตร์

In [ ]:
# ============================================================
# Setup: ติดตั้ง dependencies (สำหรับ Google Colab)
# ============================================================
!pip install anthropic python-dotenv matplotlib

In [ ]:
# ============================================================
# CONFIG: Claude API
# ============================================================

ANTHROPIC_API_KEY = ""  # ← ใส่ API key ของคุณที่นี่
MODEL = "claude-sonnet-4-6"  # หรือ claude-haiku-4-5-20251001 (ถูกกว่า)

import anthropic
import json
import re

client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)

def call_llm(prompt, system_message="คุณกำลังเล่นเป็นตัวละคร ตัดสินใจตามนิสัย ตอบเป็น JSON เท่านั้น ห้ามมีข้อความอื่น"):
    """เรียก Claude API แล้วดึง JSON จาก response"""
    print("  → กำลังส่ง prompt ไป Claude...")
    response = client.messages.create(
        model=MODEL,
        max_tokens=2048,
        system=system_message,
        messages=[{"role": "user", "content": prompt}]
    )
    text = response.content[0].text
    print(f"  → ได้ response {len(text)} characters")

    # ลอง parse JSON ตรงๆ ก่อน
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass

    # ถ้าไม่ได้ ลองหา JSON ใน code fence (รองรับ nested braces)
    code_match = re.search(r'```(?:json)?\s*\n?([\s\S]*?)\n?```', text)
    if code_match:
        try:
            return json.loads(code_match.group(1))
        except json.JSONDecodeError:
            pass

    # ลองหา JSON object (จาก { แรกถึง } สุดท้าย)
    first_brace = text.find('{')
    last_brace = text.rfind('}')
    if first_brace != -1 and last_brace != -1 and last_brace > first_brace:
        try:
            return json.loads(text[first_brace:last_brace + 1])
        except json.JSONDecodeError:
            pass

    raise ValueError(f"ไม่สามารถดึง JSON จาก response ได้:\n{text[:500]}")

print(f"Claude API พร้อมใช้งาน (model: {MODEL})")

---
# Part 1: Memory Stream

ระบบความทรงจำที่เก็บทุกประสบการณ์ของ agent

### สูตร Retrieval: `score = recency × importance × relevance`

| ส่วน | สูตร | ความหมาย |
|------|------|----------|
| **Recency** | `0.99 ^ hours` | ยิ่งเก่า ยิ่งจาง |
| **Importance** | `importance / 10` | LLM ให้คะแนน 1-10 |
| **Relevance** | `substring matching` | ตรงกับ query แค่ไหน |

In [ ]:
import json
import random
from datetime import datetime, timedelta
from dataclasses import dataclass, field
from typing import Optional


@dataclass
class Memory:
    """ความทรงจำหนึ่งอันของ Generative Agent"""
    content: str
    created_at: datetime
    importance: int              # 1-10
    memory_type: str = "observation"  # observation / reflection / plan
    related_store: Optional[str] = None
    access_count: int = 0

    def __repr__(self):
        return f"Memory('{self.content[:40]}...', imp={self.importance}, type={self.memory_type})"


class MemoryStream:
    """ระบบความทรงจำ — ค้นหาด้วย score = recency × importance × relevance"""
    
    RECENCY_DECAY = 0.99
    
    def __init__(self, agent_name: str):
        self.agent_name = agent_name
        self.memories: list[Memory] = []
    
    def add(self, content, importance, memory_type="observation",
            created_at=None, related_store=None):
        mem = Memory(content=content, created_at=created_at or datetime.now(),
                     importance=importance, memory_type=memory_type,
                     related_store=related_store)
        self.memories.append(mem)
        return mem
    
    def retrieve(self, query, top_k=5, current_time=None):
        if current_time is None:
            current_time = datetime.now()
        results = []
        for mem in self.memories:
            hours = (current_time - mem.created_at).total_seconds() / 3600
            recency = self.RECENCY_DECAY ** hours
            importance = mem.importance / 10.0
            # substring matching (รองรับภาษาไทย)
            query_keywords = set(query.lower().split())
            content_lower = mem.content.lower()
            if not query_keywords:
                relevance = 0
            else:
                matching = sum(1 for word in query_keywords if word in content_lower)
                relevance = min(1.0, matching / len(query_keywords))
            total = recency * importance * relevance
            results.append((mem, {"recency": recency, "importance": importance,
                                   "relevance": relevance, "total": total}))
        results.sort(key=lambda x: x[1]["total"], reverse=True)
        return results[:top_k]
    
    def format_for_prompt(self, results):
        if not results:
            return "ไม่มีความทรงจำที่เกี่ยวข้อง"
        return "\n".join(f"- {mem.content}" for mem, _ in results)


print("Memory + MemoryStream classes พร้อมใช้งาน!")

### ทดลอง Memory Stream

สร้าง memories ให้ **สมศรี** แล้วทดลอง query ดู

In [ ]:
somsri = MemoryStream(agent_name="สมศรี")
NOW = datetime(2024, 1, 13, 10, 0)  # วันเสาร์ 10:00 น.

# เพิ่ม memories
memories_data = [
    ("ไป Lotus's ซื้อของ พนักงานช่วยหาของดีมาก บริการประทับใจ", 7, "observation", 14*24, "Lotus's"),
    ("ซื้อของที่ Makro ราคาถูกแต่ต้องซื้อเยอะ เก็บไม่หมด", 5, "observation", 13*24, "Makro"),
    ("Lotus's มีโปรลดนม Dutch Mill 20% ซื้อมา 2 กล่อง", 6, "observation", 7*24, "Lotus's"),
    ("ไป Big C วันเสาร์ คนเยอะมาก ต่อคิวนาน ไม่ชอบเลย", 7, "observation", 7*24+2, "Big C"),
    ("Big C มีของครบกว่า Lotus's โดยเฉพาะผงซักฟอก หลายยี่ห้อ", 5, "reflection", 6*24, "Big C"),
    ("ทำอาหารเย็นให้ครอบครัว ใช้ไข่หมด 10 ฟอง", 4, "observation", 5*24, None),
    ("ลูกบอกว่าอยากกินนม Dutch Mill สตรอเบอร์รี่", 5, "observation", 4*24, None),
    ("ดูใบปลิว Big C เห็นโปรลดนม 30% สัปดาห์นี้", 7, "observation", 3*24, "Big C"),
    ("ผงซักฟอกเหลือน้อยแล้ว ต้องซื้อเพิ่ม", 6, "observation", 3*24, None),
    ("เช็คตู้เย็น นมเหลือ 1 กล่อง ต้องซื้อเพิ่มแน่ๆ", 8, "observation", 24, None),
    ("ไข่หมดไปเมื่อวาน ต้องซื้อด้วย", 7, "observation", 26, None),
    ("CJ More ใกล้บ้านที่สุด เดิน 5 นาทีถึง แต่ของไม่ค่อยครบ", 5, "reflection", 20, "CJ More"),
    ("วันนี้วันเสาร์ ไม่รีบ อากาศดี น่าออกไปซื้อของ", 4, "observation", 2, None),
    ("สามีบอกว่าอยากกินส้มตำ ต้องซื้อมะละกอด้วย", 5, "observation", 1, None),
    ("ควรไปซื้อของวันนี้ เพราะนมกับไข่เหลือน้อย", 8, "plan", 0.5, None),
]

for content, imp, mtype, hours_ago, store in memories_data:
    somsri.add(content, imp, mtype,
              created_at=NOW - timedelta(hours=hours_ago),
              related_store=store)

print(f"สมศรีมี {len(somsri.memories)} memories")

# ทดลอง query
query = "นม โปรโมชั่น ลดราคา"
print(f"\nQuery: \"{query}\"")
print("=" * 70)

results = somsri.retrieve(query, top_k=5, current_time=NOW)
for i, (mem, scores) in enumerate(results, 1):
    bar = "█" * int(scores['total'] * 50)
    print(f"  #{i} [score={scores['total']:.4f}] {mem.content}")
    print(f"     rec={scores['recency']:.3f} × imp={scores['importance']:.1f} × rel={scores['relevance']:.2f}")
    print(f"     {bar}")

---
# Part 2: Cognitive Loop

วงจรการคิด 5 ขั้นตอน:

```
PERCEIVE → RETRIEVE → PLAN → EXECUTE → REFLECT
(สังเกต)   (จำ)      (คิด)  (ทำ)      (สรุป)
                       ↑                  │
                       │  Claude LLM      │
                       ↓                  ↓
                  ┌──────────────┐
                  │Memory Stream │
                  └──────────────┘
```

In [ ]:
# ============================================================
# Customer Profile + Stores
# ============================================================

AGENT = {
    "name": "สมศรี",
    "age": 45,
    "job": "แม่บ้าน",
    "financial": {
        "monthly_income": 35000,
        "living_situation": "พอมีพอใช้ เก็บออมได้บ้าง",
        "financial_goal": "เก็บเงินให้ลูกเรียนมหาลัย"
    },
    "household": {"size": 4, "type": "บ้านเดี่ยว อยู่กับสามีและลูก 2 คน"},
    "personality": {
        "price_sensitivity": 0.8,
        "brand_loyalty": 0.6,
        "quality_preference": 0.5,
        "convenience_preference": 0.4,
        "promotion_attraction": 0.9,
        "impulse_buying": 0.3
    },
    "innate_traits": ["ใจเย็น", "ละเอียดรอบคอบ", "ชอบเปรียบเทียบราคา"],
    "routines": ["ดูใบปลิวโปรโมชั่นทุกสัปดาห์", "ไปซื้อของสดตลาดเช้าวันอาทิตย์", "ทำกับข้าวให้ครอบครัวทุกวัน"],
    "pain_points": ["ไม่ชอบร้านที่จอดรถยาก", "หงุดหงิดถ้าของที่ต้องการหมด", "ไม่ชอบพนักงานไม่สุภาพ"],
    "background": "แม่บ้านที่ดูแลครอบครัว 4 คน ชอบดูใบปลิวโปรโมชั่นทุกสัปดาห์ ละเอียดรอบคอบเรื่องราคา"
}

STORES = [
    {"name": "CJ More", "type": "supermarket",
     "distance": "ใกล้บ้านมาก เดิน 5 นาที",
     "promos": ["ไข่ลด 20%", "ผงซักฟอกลด 15%"],
     "note": "ร้านประจำละแวกบ้าน พนักงานรู้จักหน้า บริการดี ของสดไม่ค่อยครบ"},
    {"name": "Big C", "type": "hypermarket",
     "distance": "ขับรถ 20 นาที",
     "promos": ["นม Dutch Mill ลด 30%", "ซื้อนม 3 แถม 1", "ผงซักฟอกซื้อ 2 ลด 50%"],
     "note": "ของครบมาก โปรเยอะ แต่วันเสาร์คนแน่นมาก จอดรถยาก"},
    {"name": "7-Eleven", "type": "convenience",
     "distance": "ใต้คอนโด เดิน 1 นาที",
     "promos": ["นมลด 10%", "ขนมปังลด 15%", "กาแฟลด 50%"],
     "note": "สะดวกที่สุด เปิด 24 ชม. แต่ราคาแพงกว่าร้านอื่น ของสดมีน้อย"},
    {"name": "Makro", "type": "wholesale",
     "distance": "ขับรถ 30 นาที",
     "promos": ["ไข่แผงใหญ่ราคาส่ง ถูกมาก", "ข้าวสาร 5 กก. ลด 25%"],
     "note": "ราคาถูกที่สุด แต่ต้องซื้อปริมาณเยอะ เหมาะกับครอบครัวใหญ่"},
]

print(f"Agent: {AGENT['name']} ({AGENT['job']}, อายุ {AGENT['age']})")
print(f"ร้านค้า: {len(STORES)} ร้าน")

In [ ]:
# ============================================================
# STEP 1: PERCEIVE — สังเกตสิ่งรอบตัว (ไม่ใช้ LLM)
# ============================================================

day_names = {0: "จันทร์", 1: "อังคาร", 2: "พุธ", 3: "พฤหัส",
             4: "ศุกร์", 5: "เสาร์", 6: "อาทิตย์"}

perception = {
    "day": day_names[NOW.weekday()],
    "time": NOW.strftime("%H:%M"),
    "shopping_list": [
        {"item": "นม", "urgency": "high", "reason": "เหลือ 1 กล่อง"},
        {"item": "ไข่", "urgency": "high", "reason": "หมดแล้ว"},
        {"item": "ผงซักฟอก", "urgency": "medium", "reason": "เหลือน้อย"},
        {"item": "มะละกอ", "urgency": "low", "reason": "สามีอยากกินส้มตำ"},
    ],
    "budget": 3000,
    "mood": "สบายๆ ไม่รีบ",
    "available_stores": STORES,
    "query": "ซื้อของ ร้าน นม ไข่ โปรโมชั่น ลดราคา"
}

print("╔" + "═" * 58 + "╗")
print("║  STEP 1: PERCEIVE (สังเกต) — ไม่ใช้ LLM                  ║")
print("╚" + "═" * 58 + "╝")
print(f"\n  วัน: {perception['day']}  เวลา: {perception['time']}  อารมณ์: {perception['mood']}")
print(f"  งบ: {perception['budget']:,} บาท")
print(f"\n  ของที่ต้องซื้อ:")
for item in perception['shopping_list']:
    icon = {"high": "🔴", "medium": "🟡", "low": "🟢"}.get(item['urgency'], "⚪")
    print(f"    {icon} {item['item']} - {item['reason']}")
print(f"\n  ร้านค้า:")
for store in STORES:
    print(f"    - {store['name']} ({store['type']}) | {store['distance']}")
    print(f"      โปร: {', '.join(store['promos'])}")
    print(f"      หมายเหตุ: {store['note']}")

In [ ]:
# ============================================================
# STEP 2: RETRIEVE — ดึงความทรงจำ (ไม่ใช้ LLM)
# ============================================================

memory_stream = somsri  # ใช้ MemoryStream ที่สร้างไว้แล้ว
query = perception['query']
retrieved = memory_stream.retrieve(query, top_k=5, current_time=NOW)

print("╔" + "═" * 58 + "╗")
print("║  STEP 2: RETRIEVE (ดึงความทรงจำ)                         ║")
print("╚" + "═" * 58 + "╝")
print(f"\n  Query: \"{query}\"")
print(f"  จาก {len(memory_stream.memories)} memories → top {len(retrieved)}")
for i, (mem, scores) in enumerate(retrieved, 1):
    print(f"  #{i} [score={scores['total']:.4f}] {mem.content}")

memories_text = memory_stream.format_for_prompt(retrieved)

In [ ]:
# ============================================================
# STEP 3: PLAN — Claude ตัดสินใจ!
# ============================================================

def build_prompt(agent, perception, memories_text):
    p = agent['personality']
    fin = agent['financial']
    hh = agent['household']
    stores_text = []
    for s in perception['available_stores']:
        stores_text.append(f"- {s['name']} ({s['type']}) — {s.get('distance', '')}")
        stores_text.append(f"  โปร: {', '.join(s['promos'])}")
        if s.get('note'):
            stores_text.append(f"  หมายเหตุ: {s['note']}")
    stores_text = "\n".join(stores_text)
    shopping_text = "\n".join(f"- {it['item']} ({it['reason']}) [ความเร่งด่วน: {it['urgency']}]" for it in perception['shopping_list'])

    return f"""คุณกำลังเล่นเป็นตัวละครชื่อ \"{agent['name']}\"

[ตัวตนของฉัน]
- อายุ {agent['age']} ปี อาชีพ {agent['job']}
- รายได้ครอบครัว {fin['monthly_income']:,} บาท/เดือน
- สถานะการเงิน: {fin['living_situation']}
- เป้าหมายการเงิน: {fin['financial_goal']}
- มีสมาชิกในบ้าน {hh['size']} คน ({hh['type']})
- {agent['background']}

[นิสัยของฉัน (personality traits)]
- ความไวต่อราคา: {p['price_sensitivity']*10:.0f}/10 (ยิ่งสูง = ดูราคามาก)
- ความภักดีต่อแบรนด์/ร้านประจำ: {p['brand_loyalty']*10:.0f}/10
- ชอบของดี/คุณภาพ: {p['quality_preference']*10:.0f}/10
- ต้องการความสะดวก: {p['convenience_preference']*10:.0f}/10 (ยิ่งสูง = ยิ่งไม่อยากไปไกล)
- ชอบโปรโมชั่น: {p['promotion_attraction']*10:.0f}/10 (ยิ่งสูง = หลงโปรง่าย)
- ซื้อของตามใจ: {p['impulse_buying']*10:.0f}/10

[ลักษณะนิสัยโดยกำเนิด]
{chr(10).join(f'- {{t}}' for t in agent.get('innate_traits', []))}

[กิจวัตรประจำ]
{chr(10).join(f'- {{r}}' for r in agent.get('routines', []))}

[สิ่งที่ไม่ชอบ]
{chr(10).join(f'- {{pp}}' for pp in agent.get('pain_points', []))}

[ความทรงจำที่เกี่ยวข้อง]
{memories_text}

[สถานการณ์ตอนนี้]
- วัน{perception['day']} เวลา {perception['time']} ({perception['mood']})
- งบประมาณ: {perception['budget']:,} บาท

[ของที่ต้องซื้อ]
{shopping_text}

[ร้านค้าที่เลือกได้]
{stores_text}

---

จากข้อมูลทั้งหมด ให้ตัดสินใจว่าจะไปซื้อของที่ไหน
**ตัดสินใจตามนิสัย personality traits ของตัวละครเป็นหลัก**
คิดทีละขั้นตอน แล้วสรุปการตัดสินใจ

ตอบในรูปแบบ JSON:
{{{{
    "thinking_steps": ["ความคิดที่ 1", "ความคิดที่ 2", ...],
    "decision": {{{{
        "action": "go_shopping" หรือ "stay_home",
        "store": "ชื่อร้าน",
        "items_to_buy": ["รายการ1", "รายการ2"],
        "estimated_spend": จำนวนเงินที่คาดว่าจะใช้,
        "reasoning": "สรุปเหตุผลสั้นๆ"
    }}}}
}}}}"""


prompt = build_prompt(AGENT, perception, memories_text)

print("╔" + "═" * 58 + "╗")
print("║  STEP 3: PLAN (วางแผน) — ใช้ Claude!                     ║")
print("╚" + "═" * 58 + "╝")
print(f"\n  Prompt length: {len(prompt)} characters")

plan = call_llm(prompt)

print("\n  Thinking Steps:")
for i, step in enumerate(plan['thinking_steps'], 1):
    print(f"    {i}. {step}")

d = plan['decision']
print(f"\n  การตัดสินใจ:")
print(f"    ร้าน: {d.get('store', 'N/A')}")
print(f"    ของที่จะซื้อ: {', '.join(d.get('items_to_buy', []))}")
print(f"    งบประมาณ: {d.get('estimated_spend', 'N/A')} บาท")
print(f"    เหตุผล: {d['reasoning']}")

In [ ]:
# ============================================================
# STEP 4: EXECUTE — ลงมือทำ (ไม่ใช้ LLM)
# ============================================================

print("╔" + "═" * 58 + "╗")
print("║  STEP 4: EXECUTE (ลงมือทำ) — ไม่ใช้ LLM                  ║")
print("╚" + "═" * 58 + "╝")

d = plan['decision']
store_name = d.get('store', 'Unknown')
items = d.get('items_to_buy', [])
spend = d.get('estimated_spend', 0)

print(f"\n  สมศรี กำลังเดินทางไป {store_name}...")
print(f"  ถึงร้านแล้ว! กำลังซื้อของ:")
for item in items:
    print(f"    ✓ {item}")
print(f"\n  จ่ายเงิน: {spend:,} บาท")
print(f"  งบเหลือ: {perception['budget'] - spend:,} บาท")

In [ ]:
# ============================================================
# STEP 5: REFLECT — Claude สรุปบทเรียน!
# ============================================================

reflect_prompt = f"""คุณคือ \"{AGENT['name']}\" เพิ่งไปซื้อของ

สิ่งที่ทำ: {d['reasoning']}
ร้าน: {d.get('store', 'N/A')}
ของที่ซื้อ: {', '.join(d.get('items_to_buy', []))}

จากประสบการณ์นี้ สร้าง "ความทรงจำใหม่" สั้นๆ 1-2 ประโยค
และให้คะแนนความสำคัญ 1-10 (10 = จะจำไปนานมาก จะเปลี่ยนพฤติกรรมเลย)

ตอบเป็น JSON:
{{
    "new_memory": "ประโยคความทรงจำใหม่",
    "importance": 1-10
}}"""

print("╔" + "═" * 58 + "╗")
print("║  STEP 5: REFLECT (สรุปบทเรียน) — ใช้ Claude!             ║")
print("╚" + "═" * 58 + "╝")

reflection = call_llm(reflect_prompt, "สร้างความทรงจำสั้นๆ จากประสบการณ์ ตอบเป็น JSON")

print(f"\n  New Memory: \"{reflection['new_memory']}\"")
print(f"  Importance: {reflection['importance']}/10")

# เพิ่มเข้า Memory Stream
memory_stream.add(
    content=reflection['new_memory'],
    importance=reflection['importance'],
    memory_type="reflection",
    created_at=NOW + timedelta(hours=1),
    related_store=d.get('store')
)

print(f"\n  → เก็บเข้า Memory Stream แล้ว!")
print(f"  → ตอนนี้สมศรีมี {len(memory_stream.memories)} memories")
print(f"  → ครั้งหน้าที่ตัดสินใจ memory นี้จะถูกดึงมาใช้!")

---
# Part 3: Multi-Agent Retail Simulation

จำลอง **3 ลูกค้า** ที่มี personality ต่างกัน ในสถานการณ์เดียวกัน

ดูว่า Claude ตัดสินใจให้แต่ละคนต่างกันอย่างไร!

In [ ]:
# ============================================================
# Customer Profiles (3 คนที่แตกต่างกัน)
# ============================================================
import matplotlib.pyplot as plt
import numpy as np

CUSTOMERS = [
    {
        "name": "สมศรี",
        "age": 45, "job": "แม่บ้าน",
        "household": {"size": 4, "type": "บ้านเดี่ยว อยู่กับสามีและลูก 2 คน"},
        "financial": {"monthly_income": 35000, "living_situation": "พอมีพอใช้ เก็บออมได้บ้าง",
                      "financial_goal": "เก็บเงินให้ลูกเรียนมหาลัย"},
        "personality": {
            "price_sensitivity": 0.8, "brand_loyalty": 0.6, "quality_preference": 0.5,
            "convenience_preference": 0.4, "promotion_attraction": 0.9, "impulse_buying": 0.3
        },
        "innate_traits": ["ใจเย็น", "ละเอียดรอบคอบ", "ชอบเปรียบเทียบราคา"],
        "routines": ["ดูใบปลิวโปรโมชั่นทุกสัปดาห์"],
        "pain_points": ["ไม่ชอบร้านที่จอดรถยาก"],
        "background": "แม่บ้านดูแลครอบครัว 4 คน ชอบดูโปร ละเอียดเรื่องราคา",
        "label": "Somsri (Housewife, 45)"
    },
    {
        "name": "สมชาย",
        "age": 32, "job": "โปรแกรมเมอร์",
        "household": {"size": 1, "type": "คอนโด อยู่คนเดียว"},
        "financial": {"monthly_income": 80000, "living_situation": "สบาย ไม่ต้องคิดเรื่องเงิน",
                      "financial_goal": "เก็บเงินลงทุน"},
        "personality": {
            "price_sensitivity": 0.3, "brand_loyalty": 0.4, "quality_preference": 0.8,
            "convenience_preference": 0.9, "promotion_attraction": 0.2, "impulse_buying": 0.6
        },
        "innate_traits": ["ใจร้อน", "ไม่ชอบเสียเวลา", "ชอบเทคโนโลยี"],
        "routines": ["สั่งของออนไลน์บ่อย"],
        "pain_points": ["เกลียดการต่อคิวนาน", "ไม่ชอบขับรถไปไกล"],
        "background": "โปรแกรมเมอร์อยู่คอนโดคนเดียว ไม่มีเวลา ชอบความสะดวก 7-Eleven ใต้คอนโดคือที่ซื้อของประจำ",
        "label": "Somchai (Programmer, 32)"
    },
    {
        "name": "ลุงสมบัติ",
        "age": 62, "job": "ข้าราชการเกษียณ",
        "household": {"size": 2, "type": "บ้านเดี่ยว อยู่กับภรรยา"},
        "financial": {"monthly_income": 15000, "living_situation": "ต้องประหยัด ใช้เงินบำนาญ",
                      "financial_goal": "มีเงินพอใช้จนแก่"},
        "personality": {
            "price_sensitivity": 0.95, "brand_loyalty": 0.8, "quality_preference": 0.4,
            "convenience_preference": 0.3, "promotion_attraction": 0.6, "impulse_buying": 0.1
        },
        "innate_traits": ["ประหยัด", "มีระเบียบ", "ยึดติดของเดิม"],
        "routines": ["ไป CJ More ร้านประจำทุกสัปดาห์"],
        "pain_points": ["ไม่ชอบร้านที่แพงเกินจำเป็น", "ไม่ชอบเปลี่ยนร้าน"],
        "background": "ข้าราชการเกษียณ อยู่กับภรรยา ประหยัดมาก ภักดีต่อร้านประจำ CJ More มาก ไปทุกสัปดาห์จนพนักงานจำชื่อได้",
        "label": "Sombat (Retiree, 62)"
    }
]

trait_labels = {
    "price_sensitivity": "Price Sensitivity",
    "brand_loyalty": "Brand Loyalty",
    "quality_preference": "Quality",
    "convenience_preference": "Convenience",
    "promotion_attraction": "Promotion",
    "impulse_buying": "Impulse"
}

# === Radar Chart ===
categories = list(trait_labels.values())
N = len(categories)
angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
angles += angles[:1]

colors = ['#e74c3c', '#3498db', '#2ecc71']

fig, axes = plt.subplots(1, 3, figsize=(16, 5), subplot_kw=dict(polar=True))
fig.suptitle('Personality Traits Comparison', fontsize=16, fontweight='bold', y=1.02)

for idx, (customer, color) in enumerate(zip(CUSTOMERS, colors)):
    ax = axes[idx]
    values = [customer['personality'][k] for k in trait_labels.keys()]
    values += values[:1]

    ax.fill(angles, values, alpha=0.25, color=color)
    ax.plot(angles, values, 'o-', linewidth=2, color=color, markersize=6)
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(categories, size=8)
    ax.set_ylim(0, 1)
    ax.set_yticks([0.2, 0.4, 0.6, 0.8, 1.0])
    ax.set_yticklabels(['0.2', '0.4', '0.6', '0.8', '1.0'], size=7, color='gray')
    ax.set_title(customer['label'], size=10, fontweight='bold', pad=15)

plt.tight_layout()
plt.show()

# === Grouped Bar Chart ===
fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(N)
width = 0.25

for idx, (customer, color) in enumerate(zip(CUSTOMERS, colors)):
    values = [customer['personality'][k] for k in trait_labels.keys()]
    bars = ax.bar(x + idx * width, values, width, label=customer['label'],
                  color=color, alpha=0.8, edgecolor='white')
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                f'{val}', ha='center', va='bottom', fontsize=8, fontweight='bold')

ax.set_xlabel('Personality Traits', fontsize=11)
ax.set_ylabel('Score (0-1)', fontsize=11)
ax.set_title('Personality Comparison: 3 Agents', fontsize=14, fontweight='bold')
ax.set_xticks(x + width)
ax.set_xticklabels(categories, fontsize=9)
ax.set_ylim(0, 1.15)
ax.legend(fontsize=9)
ax.grid(axis='y', alpha=0.3)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# รัน Simulation: 3 agents ใช้ Claude ตัดสินใจ!
# ============================================================

# Memories เฉพาะของแต่ละ agent
AGENT_MEMORIES = {
    "สมศรี": [
        ("ดูใบปลิว Big C เห็นโปรลดนม 30% สัปดาห์นี้ น่าสนใจมาก", 7, 3*24),
        ("ไป Big C วันเสาร์ที่แล้ว คนเยอะมาก ต่อคิวนาน แต่โปรดีมาก", 6, 7*24),
        ("CJ More ใกล้บ้าน แต่โปรไม่ค่อยดีเท่า hypermarket", 4, 10*24),
        ("ซื้อนมที่ Big C ลดราคาได้ประหยัดเยอะ คุ้มค่าที่ไปไกลหน่อย", 7, 14*24),
    ],
    "สมชาย": [
        ("ซื้อนมที่ 7-Eleven ใต้คอนโดเมื่อวาน สะดวกดี ไม่ต้องขับรถไปไหน", 6, 24),
        ("ลองไป Big C เสาร์ที่แล้ว รถติดมาก เสียเวลา 2 ชั่วโมง ไม่คุ้ม", 8, 7*24),
        ("ปกติซื้อของที่ 7-Eleven ใต้คอนโด แพงนิดหน่อยแต่สะดวกมาก", 5, 3*24),
        ("สั่ง Makro ออนไลน์ได้ แต่ต้องรอ 2-3 วัน ไม่ทันใช้", 4, 5*24),
    ],
    "ลุงสมบัติ": [
        ("ไป CJ More เหมือนทุกสัปดาห์ พนักงานจำชื่อได้ บริการดีมาก", 7, 7*24),
        ("CJ More มีไข่ลด 20% สัปดาห์นี้ ดีเลย", 6, 2*24),
        ("ไม่เคยไป Big C เลย ไกลบ้าน ไม่คุ้มค่าน้ำมัน", 5, 10*24),
        ("CJ More ใกล้บ้าน เดิน 5 นาทีถึง ไปมาสะดวก", 5, 14*24),
    ],
}

print("=" * 70)
print("  RETAIL SIMULATION — 3 Agents x Claude API")
print("  สถานการณ์เดียวกัน แต่ตัดสินใจต่างกัน!")
print("=" * 70)

sim_results = []

for customer in CUSTOMERS:
    name = customer['name']
    print(f"\n{'─'*70}")
    print(f"  Agent: {name} ({customer['job']})")
    print(f"{'─'*70}")

    # สร้าง memory stream เฉพาะของแต่ละ agent
    ms = MemoryStream(agent_name=name)
    for content, imp, hours_ago in AGENT_MEMORIES.get(name, []):
        ms.add(content, imp, "observation", NOW - timedelta(hours=hours_ago))
    ms.add("ของในบ้านเหลือน้อย ต้องไปซื้อ", 7, "plan", NOW - timedelta(hours=1))

    mem_text = ms.format_for_prompt(ms.retrieve("ซื้อของ ร้าน นม ไข่", top_k=5, current_time=NOW))

    print(f"  [PERCEIVE] สังเกตสิ่งรอบตัว...")
    print(f"  [RETRIEVE] ดึงความทรงจำ {len(ms.memories)} อัน...")

    print(f"  [PLAN] Claude กำลังคิดให้ {name}...")
    agent_prompt = build_prompt(customer, perception, mem_text)
    agent_plan = call_llm(agent_prompt)

    for i, step in enumerate(agent_plan['thinking_steps'], 1):
        print(f"    {i}. {step}")

    ad = agent_plan['decision']
    print(f"\n  [EXECUTE] -> ไป {ad.get('store', 'N/A')}")
    print(f"     ซื้อ: {', '.join(ad.get('items_to_buy', []))}")
    print(f"     จ่าย: {ad.get('estimated_spend', 0):,} บาท")
    print(f"  [REASON] {ad['reasoning']}")

    sim_results.append({
        "name": name,
        "store": ad.get('store', 'N/A'),
        "spend": ad.get('estimated_spend', 0),
        "items": ad.get('items_to_buy', []),
        "reasoning": ad['reasoning']
    })

In [ ]:
# ============================================================
# Results Comparison
# ============================================================

# === Text summary (Thai) ===
print("=" * 70)
print("  COMPARISON: Results")
print("=" * 70)
for r in sim_results:
    print(f"\n  {r['name']}:")
    print(f"    Store: {r['store']}  |  Spend: {r['spend']:,} baht")
    print(f"    Items: {', '.join(r['items'])}")
    print(f"    Reason: {r['reasoning']}")

# === Charts (English) ===
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors_bar = ['#e74c3c', '#3498db', '#2ecc71']

# Map Thai names to English for chart
name_map = {c['name']: c['label'] for c in CUSTOMERS}

# 1. Spending comparison
ax1 = axes[0]
labels = [name_map.get(r['name'], r['name']) for r in sim_results]
spends = [r['spend'] for r in sim_results]
stores = [r['store'] for r in sim_results]

bars = ax1.barh(labels, spends, color=colors_bar, edgecolor='white', height=0.5)
for bar, store, spend in zip(bars, stores, spends):
    ax1.text(bar.get_width() + 20, bar.get_y() + bar.get_height()/2,
             f'{spend:,} baht @ {store}', va='center', fontsize=10, fontweight='bold')
ax1.set_xlabel('Spending (baht)', fontsize=11)
ax1.set_title('How much each agent spent', fontsize=13, fontweight='bold')
ax1.set_xlim(0, max(spends) * 1.6)
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)
ax1.grid(axis='x', alpha=0.3)

# 2. Key traits that drove decisions
ax2 = axes[1]
key_traits = ['promotion_attraction', 'convenience_preference', 'brand_loyalty', 'price_sensitivity']
key_labels = ['Promotion', 'Convenience', 'Brand Loyalty', 'Price Sens.']
x = np.arange(len(key_traits))
width = 0.25

for idx, (customer, color) in enumerate(zip(CUSTOMERS, colors_bar)):
    vals = [customer['personality'][t] for t in key_traits]
    ax2.bar(x + idx * width, vals, width, label=customer['label'], color=color, alpha=0.8)

ax2.set_xticks(x + width)
ax2.set_xticklabels(key_labels, fontsize=9)
ax2.set_ylabel('Score', fontsize=11)
ax2.set_title('Key traits driving store choice', fontsize=13, fontweight='bold')
ax2.set_ylim(0, 1.1)
ax2.legend(fontsize=8)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

---
## สรุป

### สิ่งที่ได้เรียนรู้

1. **Memory Stream** เก็บทุกประสบการณ์ ค้นหาด้วย `recency × importance × relevance`
2. **Cognitive Loop** 5 ขั้นตอน — Claude เป็นสมองในขั้น Plan + Reflect
3. **Personality Traits** กำหนดพฤติกรรม — คนรักโปรไป hypermarket, คนรักสะดวกไป 7-11
4. **ไม่มี if-else!** ทุกการตัดสินใจผ่าน LLM = realistic มากกว่า rule-based

### Applications
- **Marketing**: ทดสอบโปรโมชั่นก่อนลองจริง
- **Store Planning**: เข้าใจว่าลูกค้าแต่ละกลุ่มเลือกร้านอย่างไร
- **Policy Testing**: เปลี่ยนราคา/โปร → ดูว่าพฤติกรรมเปลี่ยนไหม